In [1]:
# Streamlit'i yükle (eğer kurulu değilse)
!pip install -q streamlit
!pip install -q transformers diffusers accelerate torch

import streamlit as st
import torch
from transformers import pipeline
from diffusers import StableDiffusionPipeline
from PIL import Image, ImageFilter
import os


# --- HAFTA 12: Sayfa Konfigürasyonu ve UI ---
st.set_page_config(page_title="Science & Art AI App", layout="wide")
st.title("🎨 Science and Art: Generative AI Suite")
st.sidebar.header("Haftalık Görev Menüsü")

# --- HAFTA 9: Mod Seçimi (Chat vs Art) ---
#  Chatbot ve Image Generator birleştirildi.
mode = st.sidebar.radio("Bir mod seçin:", ["Chat Mode (W5/9)", "Art Mode (W7/9/11)"])

# --- MODEL YÜKLEME (Caching ile hızlandırma) ---
@st.cache_resource
def load_chat_model():
    # Hafta 5: Ücretsiz Hugging Face modeli [cite: 2, 13]
    return pipeline("text2text-generation", model="google/flan-t5-base")

@st.cache_resource
def load_art_model():
    # Hafta 7: Stable Diffusion open-source model [cite: 2, 13]
    model_id = "runwayml/stable-diffusion-v1-5"
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
    if torch.cuda.is_available():
        pipe = pipe.to("cuda")
    return pipe

# --- CHAT MODE (Hafta 5 & 9) ---
if mode == "Chat Mode (W5/9)":
    st.header("💬 AI Chatbot")

    # HAFTA 10: Parametre Optimizasyonu
    # Sınavda sorulursa: temperature yaratıcılığı, top_p olasılık dağılımını belirler.
    with st.sidebar.expander("Prompt Ayarları (W10)"):
        temp = st.slider("Temperature (Yaratıcılık)", 0.1, 1.0, 0.7)
        max_len = st.slider("Maksimum Yanıt Uzunluğu", 20, 200, 50)

    user_query = st.text_input("Bir soru sorun:")
    if user_query:
        chat_model = load_chat_model()
        response = chat_model(user_query, max_length=max_len, temperature=temp)
        st.write("**Cevap:**", response[0]['generated_text'])

# --- ART MODE (Hafta 7, 9 & 11) ---
elif mode == "Art Mode (W7/9/11)":
    st.header("🖼️ Generative Art")

    prompt = st.text_input("Hayalinizdeki resmi tarif edin (İngilizce):", "A futuristic city in watercolor style")

    if st.button("Resim Oluştur"):
        with st.spinner("Sanat eseri hazırlanıyor..."):
            art_model = load_art_model()
            # Hafta 7: Resim üretme
            generated_image = art_model(prompt).images[0]

            # Resmi geçici olarak kaydet (Hafta 7 isterleri) [cite: 2, 9]
            if not os.path.exists("outputs"): os.makedirs("outputs")
            generated_image.save("outputs/last_generated.png")

            st.image(generated_image, caption="Orijinal Yapay Zeka Çıktısı")
            st.session_state['current_img'] = generated_image

    # --- HAFTA 11: Artistic Style Enhancement (Filtreler) ---
    #  Pillow/OpenCV kullanarak filtre ekleme.
    if 'current_img' in st.session_state:
        st.divider()
        st.subheader("Sanatsal Filtreler (W11)")
        filter_type = st.selectbox("Bir filtre seçin:", ["Orijinal", "Siyah-Beyaz", "Bulanık (Blur)", "Kenar Belirleme"])

        processed_img = st.session_state['current_img'].copy()

        if filter_type == "Siyah-Beyaz":
            processed_img = processed_img.convert("L")
        elif filter_type == "Bulanık (Blur)":
            processed_img = processed_img.filter(ImageFilter.BLUR)
        elif filter_type == "Kenar Belirleme":
            processed_img = processed_img.filter(ImageFilter.FIND_EDGES)

        st.image(processed_img, caption=f"Uygulanan Filtre: {filter_type}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 41.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
2026-01-13 18:24:52.579 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 18:24:52.580 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-13 18:24:52.677 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-01-13 1